In [30]:
import pandas as pd
import numpy as np

Part 1:

In [45]:
students = pd.read_csv("../data/raw/students.csv")
enrollments = pd.read_csv("../data/raw/enrollments.csv")
courses = pd.read_csv("../data/raw/courses.csv")

In [32]:
print(students.shape)
print(students.columns)
print(students.dtypes)
print(students.isnull().sum())


(30, 11)
Index(['student_id', 'first_name', 'last_name', 'program', 'year', 'age',
       'active', 'entry_term', 'gpa', 'credits_completed', 'advisor'],
      dtype='str')
student_id               str
first_name               str
last_name                str
program                  str
year                     str
age                  float64
active                   str
entry_term               str
gpa                  float64
credits_completed      int64
advisor                  str
dtype: object
student_id           0
first_name           0
last_name            0
program              0
year                 0
age                  2
active               0
entry_term           0
gpa                  3
credits_completed    0
advisor              1
dtype: int64


In [33]:
print(enrollments.shape)
print(enrollments.columns)
print(enrollments.dtypes)
print(enrollments.isnull().sum())


(59, 9)
Index(['enrollment_id', 'student_id', 'course_id', 'term', 'status',
       'midterm_score', 'final_score', 'project_score', 'attendance_pct'],
      dtype='str')
enrollment_id         str
student_id            str
course_id             str
term                  str
status                str
midterm_score       int64
final_score       float64
project_score     float64
attendance_pct    float64
dtype: object
enrollment_id     0
student_id        0
course_id         0
term              0
status            0
midterm_score     0
final_score       5
project_score     3
attendance_pct    2
dtype: int64


In [34]:
print(courses.shape)
print(courses.columns)
print(courses.dtypes)
print(courses.isnull().sum())


(8, 7)
Index(['course_id', 'department', 'course_number', 'course_title', 'credits',
       'level', 'delivery'],
      dtype='str')
course_id          str
department         str
course_number    int64
course_title       str
credits          int64
level              str
delivery           str
dtype: object
course_id        0
department       0
course_number    0
course_title     0
credits          0
level            0
delivery         0
dtype: int64


The Key columns are going to be student_id, course_id, and enrollment_id. These will allow us to merge the tables and find some further information out about the data. 

The most immediate concerns are the amount of null entries and the sizes of the dataframes. Within the students dataframe age has 2 null entries, gpa has 3 null entries, and advisor has 1 null entry. In enrollments, final_score has 5 null entries, project_score has 3 null entries, and attendance_pct has 2 null entries. In courses there are no null entries. Another concern is that the sizes of each data set are drastically different, with students having 30 entries, enrollements 59, and courses 8. This means when we take a deeper look at the data we might run into some issues with duplicates. 

Part 2

In [35]:
#1
students.loc[
    (students["program"] == "Mathematics") &
    (students["year"] == "1st")

]

,student_id,first_name,last_name,program,year,age,active,entry_term,gpa,credits_completed,advisor
0,S1001,Avery,Nguyen,Mathematics,1st,27.0,Yes,Fall 2023,3.51,72,Dr. Moreno
2,S1003,Noah,Garcia,Mathematics,1st,18.0,Yes,Spring 2024,2.81,18,Dr. Reed
13,S1014,Henry,Anderson,Mathematics,1st,20.0,Yes,Spring 2025,2.66,45,Dr. Shah
19,S1020,Daniel,Young,Mathematics,1st,22.0,Yes,Fall 2023,3.75,108,Dr. Shah
27,S1028,Caleb,Hill,Mathematics,1st,20.0,No,Spring 2026,3.79,108,Dr. Reed


I wanted to find all students who were in their first year of the Mathematics program. 

In [36]:
#2
students["high_gpa"] = students["gpa"] >= 3.5

High_math_gpa = students.loc[
    (students["program"] == "Mathematics") &
    (students["high_gpa"]),
    ["first_name", "last_name"]]
#High_math_gpa


I also wanted to find which Mathematics students had a "high" gpa (>3.5). I created a derived variable for those high gpa's, and then used that to filter students for those in Mathematics and those who had a high gpa.

In [37]:
#3 
high_gpa_students = students.loc[
    students["gpa"] >= 3.5,
    ["first_name", "last_name","program"]
]
#high_gpa_students

I could also use .loc to find all students with a high gpa. 

In [38]:

#4
high_gpa_list = [gpa >= 3.5 for gpa in students["gpa"]]
sum(high_gpa_list)

10

I just wanted to see how many students overall had a high gpa, and so I used list comphrehension to list all students by a True/False for whether they had a high gpa or not. I then summed those to find the total number of True entries. 

Part 3

In [49]:
#I want to get student and enrollment merged on student_id. 
student_enrollment = pd.merge(
    students,
    enrollments,
    on = "student_id",
    how = "left",
    validate = "1:m",
    indicator = True
)
student_enrollment[student_enrollment["_merge"]!="both"]
#student_enrollment[["student_id","first_name","last_name","term","enrollment_id","course_id","final_score","True"]]



,student_id,first_name,last_name,program,year,age,active,entry_term,gpa,credits_completed,advisor,enrollment_id,course_id,term,status,midterm_score,final_score,project_score,attendance_pct,_merge
58,S1030,Miles,Mitchell,Computer Science,4th,22.0,Yes,Fall 2024,NaN,36,Dr. Kim,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


Using indicator=True, I found that the student Miles Mitchell has no enrollment information.There are also alot of duplicates of students in here. This makes sense as each student can be taking multiple courses.

In [40]:

student_enrollment_courses = pd.merge(
    student_enrollment,
    courses,
    on = "course_id",
    how = "left",
    validate = "m:1",
)
#student_enrollment_courses[(student_enrollment_courses["course_title"] == "Applied Statistics")]


Part 4

In [41]:
#1 Missing data counts and percentages
student_enrollment_courses.isna().sum()



student_id           0
first_name           0
last_name            0
program              0
year                 0
age                  4
active               0
entry_term           0
gpa                  5
credits_completed    0
advisor              1
high_gpa             0
enrollment_id        1
course_id            1
term                 1
status               1
midterm_score        1
final_score          6
project_score        4
attendance_pct       3
True                 0
department           2
course_number        2
course_title         2
credits              2
level                2
delivery             2
dtype: int64

In [42]:

student_enrollment_courses.isna().mean()*100


student_id            0.000000
first_name            0.000000
last_name             0.000000
program               0.000000
year                  0.000000
age                   6.779661
active                0.000000
entry_term            0.000000
gpa                   8.474576
credits_completed     0.000000
advisor               1.694915
high_gpa              0.000000
enrollment_id         1.694915
course_id             1.694915
term                  1.694915
status                1.694915
midterm_score         1.694915
final_score          10.169492
project_score         6.779661
attendance_pct        5.084746
True                  0.000000
department            3.389831
course_number         3.389831
course_title          3.389831
credits               3.389831
level                 3.389831
delivery              3.389831
dtype: float64

In [43]:
student_enrollment_courses[["course_id","enrollment_id"]].isnull()


,course_id,enrollment_id
0,False,False
1,False,False
2,False,False
3,False,False
4,False,False
5,False,False
6,False,False
7,False,False
8,False,False
9,False,False


Row 58 has null values for both course_id and enrollment_id. 

In [44]:
student_avg_score = student_enrollment.pivot_table(
    index = "student_id",
    columns = "term",
    values = "final_score",
    aggfunc= "mean"
)
student_avg_score

term,Fall 2025,Fall 2026,Spring 2026
student_id,,,
S1001,NaN,NaN,70.666667
S1002,55.0,NaN,NaN
S1003,NaN,76.0,NaN
S1004,58.0,79.0,73.000000
S1005,55.0,NaN,57.000000
S1006,NaN,81.0,NaN
S1007,93.0,NaN,82.000000
S1008,65.0,82.0,76.500000
S1009,NaN,61.0,NaN


Replacing all the NaN values with 0 is a terrible idea, as a value of NaN in this case simply means they did not take any courses for that term. Replacing them with 0's would imply they had an average final score of 0, which is not the case. 

Part 5

In [52]:
student_enrollment_courses.to_csv("../data/processed/student_enrollment_courses.csv")

OSError: Cannot save file into a non-existent directory: '../data/processed'